In [15]:
import app.app_logic as app_logic
from app.together_api_client import TogetherMixtralClient
from app.prompt_models import RAGPrompt
from app.db_connector import get_engine
from app.icog_util import DocSummarizer
from app.transformers_util import generate_embeddings, get_util
import re
from sqlalchemy import (
    text,
)

engine = get_engine()
ts_util = get_util()

user_id="yU13Hk9BwEQiREgh91YM6EFKR7M2"
summarizer = DocSummarizer()

client = TogetherMixtralClient()


text = "What is the capital of France? How do you bake a cake? Tell me about your favorite movie."

pattern = r"(?:What|How|Tell me|Find).*\?"
questions = re.findall(pattern, text, flags=re.IGNORECASE)
print(questions)


question_text = "What is special about Phil Jackson leadership?"
docs = app_logic.search_embeddings(user_id=user_id, search_term=question_text, max_results=3, threshold=0.5)

retrieved_contexts = []
for doc in docs:
    td = app_logic.get_document_by_id(doc.id)
    summary = summarizer(td.original_text)
    retrieved_contexts.append({"doc_id": doc.id, "text": summary})

print(len(retrieved_contexts))

messages_list = RAGPrompt.get_messages(contexts=retrieved_contexts, question=question_text)
answer = await client.generate(messages=messages_list, model=RAGPrompt)
answer

2024-04-18 09:48:00 - Connecting to database using TCP
2024-04-18 09:48:00 - Connected to database using TCP
2024-04-18 09:48:00 - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7fed9d360b00>
['What is the capital of France? How do you bake a cake?']
2024-04-18 09:48:00 - Generate embeddings for term What is special about Phil Jackson leadership?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2024-04-18 09:48:00 - Embeddings for term What is special about Phil Jackson leadership? are length is 384
2024-04-18 09:48:00 - Searching for documents with embeddings closest to term What is special about Phil Jackson leadership?
2024-04-18 09:48:00 - -- Found 2 matched documents for term What is special about Phil Jackson leadership?
2024-04-18 09:48:00 - Found 1 matched documents for term What is special about Phil Jackson leadership?
1
2024-04-18 09:48:00 - Attempt 1 to generate summary
2024-04-18 09:48:03 - Response status: finished
2024-04-18 09:48:03 - Answer is not None. Stop retrying. Number of attempts 1


RAGPrompt(answer="Phil Jackson's leadership is considered special due to its unconventional nature and effectiveness. He won 11 NBA titles by leading difficult players under pressure from demanding managers. Jackson's leadership style included watching his team lose without giving instructions, holding group meditation exercises, and giving players novels as reading assignments. Five key lessons from his memoir 'Eleven Rings' include: (1) focusing on the present moment, (2) earning power and trust, (3) committing to continuous growth, (4) being resourceful, and (5) understanding that leadership is about human relations and encouraging the team to do the right thing.", document_ids_used_for_answer=[129], usage={'prompt_tokens': 747, 'completion_tokens': 227, 'total_tokens': 974, 'duration': 2811})

In [20]:
import pprint
print(type(answer))
pprint.pprint(answer.answer)
print(answer.document_ids_used_for_answer)




<class 'app.prompt_models.RAGPrompt'>
("Phil Jackson's leadership is considered special due to its unconventional "
 'nature and effectiveness. He won 11 NBA titles by leading difficult players '
 "under pressure from demanding managers. Jackson's leadership style included "
 'watching his team lose without giving instructions, holding group meditation '
 'exercises, and giving players novels as reading assignments. Five key '
 "lessons from his memoir 'Eleven Rings' include: (1) focusing on the present "
 'moment, (2) earning power and trust, (3) committing to continuous growth, '
 '(4) being resourceful, and (5) understanding that leadership is about human '
 'relations and encouraging the team to do the right thing.')
[129]


In [9]:
for c in retrieved_contexts:
    print(f"{c['doc_id']} - {c['text']}")

178 - An MVP strips out all unnecessary functionality, and includes the bare minimum amount of features in order to get feedback from potential customers. It’s an established, useful concept that, when utilised correctly, can minimise wasted time in the early stages of defining a product vision and establishing product / market fit. Rather than spending a large amount of time creating a full product and hoping for traction, you can identify the crucial functionality and build just these features, in order to determine whether to pivot, or to pursue the project. Choose the audience and scope for your MVP carefully When you’re looking at creating an MVP, you should have a hypothesis in mind that you’re looking to prove or disprove — essentially, a measurable way of saying whether your MVP succeeded or not. Many users can’t see past basic usability issues in an MVP, and will require a refined product to provide useful feedback Imagine you’ve created a micro-site to test whether your new ‘

In [14]:
messages_list = RAGPrompt.get_messages(contexts=retrieved_contexts, question=question_text)
answer = await client.generate(messages=messages_list, model=RAGPrompt)

2024-04-17 20:06:30 - Attempt 1 to generate summary
2024-04-17 20:06:34 - Response status: finished
2024-04-17 20:06:34 - Answer is not None. Stop retrying. Number of attempts 1


In [15]:
answer

RAGPrompt(answer="The term 'Minimum Valuable Experience' (MVE) is not meant to replace 'Minimum Viable Product' (MVP), but rather to emphasize the importance of creating a valuable experience for the user. The MVE concept encourages entrepreneurs to focus on delivering a product or service that provides value to the user, rather than just meeting the minimum requirements for viability. By creating an MVE, entrepreneurs can better understand their audience, tailor their product to meet the needs and preferences of their users, and ultimately build a stronger and more loyal customer base. In this sense, an MVE is a more holistic and user-centered approach to product development than the traditional MVP, which can sometimes prioritize speed and functionality over user value and experience.", usage={'prompt_tokens': 2580, 'completion_tokens': 202, 'total_tokens': 2782, 'duration': 3799})

In [14]:
import re
text = "what is the capital of France? how do you bake a cake? Tell me about your favorite movie."

pattern = r"(?:What|How|Tell me|Find).*\?"
questions = re.match(pattern, text, flags=re.IGNORECASE)
print(questions.group(0))

what is the capital of France? how do you bake a cake?
